<a href="https://colab.research.google.com/github/MichaelZimm20/xn-sitstayforever-semantic-attention/blob/main/notebooks/xn_sitstayforever_data_and_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [20]:
#--- DIRECTORIES-----
BASE_DIR = '/content/drive/MyDrive/xn-sitstayforever'
DATASET_DIR = f'{BASE_DIR}/datasets'
OUTPUTS_DIR = f'{BASE_DIR}/outputs'
CHECKPOINTS_DIR = f'{BASE_DIR}/checkpoints'
IMAGE_DIR = f'{DATASET_DIR}/product_images'

print(DATASET_DIR)

/content/drive/MyDrive/xn-sitstayforever/datasets


# Setup & Imports

In [4]:
# Installs for missing dependencies not natively supported
!pip install open-clip-torch -q

print("Open CLIP install successful!")

Open CLIP install successful!


In [21]:
#------- IMPORTS -------------
# --System Libraries---
import os
import sys
import random
from pathlib import Path

# --Data Handling---
import numpy as np
import pandas as pd

# --Visualizations---
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# --Preprocessing & Encoding---
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans



# --Image Processing---
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from tqdm import tqdm # wraps any loop and gives a live progress bar when loading the images in a loop

# --Deep Learning---
import torch
import torchvision.transforms as transforms

# --CLIP---
import open_clip

print("All Libraries have been successfully imported!")

All Libraries have been successfully imported!


# Dataset Loading

In [13]:
import os

# File path smoke test, since files are in a google drive and mounted
# for item in os.listdir(BASE_DIR):
#   print(repr(item))
print('=' * 50)
for item in os.listdir(DATASET_DIR):
  print(item)
print('=' * 50)
# check exact folder name including hidden characters
items = os.listdir(BASE_DIR)
for item in items:
  print(repr(item))

# check it path exists
print(os.path.exists(DATASET_DIR))

SSF_CV_Dataset.xlsx
pet_cv_dataset_full.xlsx
product_images
'datasets'
'outputs'
'checkpoints'
True


In [18]:
# ------ LOADING THE DATASET ------
''' The dataset consists of 54 images,
text, 2 spreadsheets which need to
be imported, analyzed and clean during
this preprocessing stage'''

# load the pet_cv_dataset.xlsx using pandas
df_images = pd.read_excel(f'{DATASET_DIR}/pet_cv_dataset_full.xlsx', sheet_name='images_cv_features')
df_products = pd.read_excel(f'{DATASET_DIR}/pet_cv_dataset_full.xlsx', sheet_name='product_summary')

# load SSF_CV_Dataset.xlsx using pandas
df_ssf = pd.read_excel(f'{DATASET_DIR}/SSF_CV_Dataset.xlsx', sheet_name='images_cv_features')
df_keywords = pd.read_excel(f'{DATASET_DIR}/SSF_CV_Dataset.xlsx', sheet_name='keywords_dataset')

# --Outputs---#
print(f'(df_images:   {df_images.shape}')
print(f'(df_products:   {df_products.shape}')
print(f'(df_ssf:   {df_ssf.shape}')
print(f'(df_keywords:   {df_keywords.shape}')

print('=' * 50)
''' Note:
df_images is outputting 145,44 instead of 54,44 to follow the 54 images. Its reading all 100 metadata rows plus the
54 rows combined. Will resolve this next section'''

(df_images:   (145, 44)
(df_products:   (100, 12)
(df_ssf:   (6, 36)
(df_keywords:   (146, 12)


' Note:\ndf_images is outputting 145,44 instead of 54,44 to follow the 54 images. Its reading all 100 metadata rows plus the \n54 rows combined. Will resolve this next section'

In [22]:
# ------ LOADING THE IMAGES FROM GOOGLE DRIVE ------
''' This section will handle loading the images from the dataset. The images are stored in a folder on
Google drive

- since the imags can come in different channel formats, RGBA, grayscale, palette-based, it would be best to unify the formats
  - this will help our other models like CLIP, GradCAM etc to interpret the the same format instead of mixed formats



Storing the images and filepaths seperately for useablilty for other libaries and models. Better for memory and speed
'''

images = {} # store image objects in empty dictionary, ordered by filename
image_paths = {} # store full image file paths by filename to ensure we have correct location

# check file name, path, extension and sort
'''tqdm adds a real-time progress bar to the loop to track execution speed
    and time for image uploads to the dictionaries '''
for filename in tqdm(sorted(os.listdir(IMAGE_DIR))):
  if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
    filepath = os.path.join(IMAGE_DIR, filename)

    try:
      img = Image.open(filepath).convert('RGB') # read the image from drive and convert pixels to RGB (to standardize color 3 color channels)
      images[filename] = img
      image_paths[filename] = filepath
    except Exception as e:
      print(f"Error loading image {filename}: {e}")
      continue

print(f'\nLoaded {len(images)} images from {IMAGE_DIR}')


100%|██████████| 54/54 [00:41<00:00,  1.31it/s]


Loaded 54 images from /content/drive/MyDrive/xn-sitstayforever/datasets/product_images
